# 厳密対角化（Exact Diagonalization）

ハミルトニアンを行列で表し、数値的に対角化する方法。近似を使わないので結果は厳密であり、どんな模型にも使える。そのかわり、行列のサイズが系の大きさとともに指数関数的に増えるので、扱えるのは小さい系に限られる。他の手法の結果を確かめる「答え合わせ役」としても重要である。

各模型への適用（行列の具体形や手計算）は、`models/<模型>/01_exact_diag.ipynb` にまとめてある。

## 1. ハミルトニアンの行列表示

スピン1/2が$n$個ある系の状態は$2^n$次元のベクトルで、ハミルトニアンは$2^n\times2^n$の行列になる。Pauli演算子の積で書かれたハミルトニアンは、各サイトの$2\times2$行列のテンソル積（クロネッカー積）で作れる。サイト$i$だけに演算子$O$が作用する行列は

\begin{equation}
O_i = I\otimes\cdots\otimes I\otimes \underbrace{O}_{i\text{番目}}\otimes I\otimes\cdots\otimes I
\end{equation}

であり、2サイトにまたがる項$O_iO_j$はその積で表せる。このリポジトリでは`qmb.operators.site_op(op, i, n)`がこの行列を返す（量子ビットの並び順は[docs/conventions.md](../docs/conventions.md)を参照）。

## 2. 基底エネルギーと固有状態

エルミート行列$H$を対角化すると

\begin{equation}
H = \sum_{m} E_m\ket{m}\bra{m},\quad E_0\le E_1\le\cdots
\end{equation}

となる。最小の固有値$E_0$が基底エネルギー、対応する固有ベクトル$\ket{0}$が基底状態である。励起エネルギー（ギャップ）$E_1 - E_0$なども同時に求まる。

## 3. 時間発展と期待値

シュレディンガー方程式

\begin{equation}
i\frac{d}{dt}\ket{\psi(t)} = H\ket{\psi(t)} \Rightarrow \ket{\psi(t)} = e^{-iHt}\ket{\psi_0}
\end{equation}

の解は、初期状態を固有基底で展開すると簡単に求まる。

\begin{equation}
\ket{\psi_0} = \sum_m c_m\ket{m},\quad c_m = \bra{m}\ket{\psi_0}
\Rightarrow
\ket{\psi(t)} = \sum_m e^{-iE_mt}c_m\ket{m}
\end{equation}

各固有状態には位相$e^{-iE_mt}$がつくだけなので、一度対角化すればどの時刻の状態もすぐに求まる。観測量$A$の期待値は

\begin{equation}
\langle A\rangle(t) = \bra{\psi(t)}A\ket{\psi(t)} = \sum_{m, m'} c_m^{*}c_{m'} e^{i(E_m - E_{m'})t}\bra{m}A\ket{m'}
\end{equation}

である。期待値はエネルギー差$E_m - E_{m'}$の周波数で振動する成分の和になる。

## 4. 計算コストと工夫

- **メモリ**：複素数の$2^n\times2^n$行列を保持するには$16\times4^n$バイトが必要で、$n=12$で約270 MB、$n=14$で約4 GBになる。
- **計算時間**：全固有値を求める対角化は$O(2^{3n})$で、$n$が1増えるごとに約8倍になる。
- **疎行列とLanczos法**：局所的なハミルトニアンの行列はほとんどの成分が0（疎行列）なので、疎行列として保持し、基底状態など少数の固有値だけをLanczos法などで求めると、$n\sim20$〜$30$程度まで扱える。
- **対称性によるブロック対角化**：$H$と交換する保存量（パリティ、粒子数、運動量など）があると、$H$はその固有値ごとのブロックに分かれる。ブロックごとに対角化すれば、行列のサイズを小さくできる。

## 5. このリポジトリでの関数

| 関数 | 内容 |
|---|---|
| `qmb.operators.site_op(op, i, n)` | サイト$i$だけに`op`が作用する行列 |
| `qmb.ed.ground_state(H)` | 基底エネルギー$E_0$と基底状態 |
| `qmb.ed.time_evolve_expectation(H, psi0, A, ts)` | 各時刻の期待値$\langle A\rangle(t)$ |
| `qmb.observables.energy_gap(H)` | ギャップ$E_1 - E_0$ |

---
**次へ**: [02 Jordan-Wigner変換](02_jordan_wigner.ipynb)